In [ ]:
# Notebook: 01a_output_validation.ipynb
#
# Objective
# - Bring the six review items into one report.
# - Generate the missing locked-quote detail.
# - Carry forward Raymond's completed reviews.
# - Record the remaining answers and contributions.
#
# Prerequisites
# - Notebook 01 completed successfully using etl_01_v3.
# - Open this notebook from the repository or notebooks folder.
# - Use the current Python environment.
#
# Inputs
# - Saved observed and analysis Parquet files.
# - Notebook 01 execution summary and exclusion windows.
# - Notebook 00 holiday context and verified closure records.
#
# Outputs
# - 01a_output_validation.html.
# - Detailed locked-quote CSV.
# - Dead-zone and boundary-anchor CSVs.
# - Review answers and validation summary.
# - Contributions and project records after the open decisions are answered.
#
# Completed human reviews
# - Remaining dead zones are accepted as valid exclusions.
# - Holiday treatment is accepted with its evidence limitations.
# - Right-anchor availability was reviewed in examples.
# - Zero-size observations may support market analysis.
# - Zero-size observations are excluded from P&L modeling inputs.
#
# Open decisions
# - How to treat the locked quotes after inspecting them.
# - Whether excluded-hour quotes may remain interpolation anchors.
#
# Remaining limitation
# - Absolute size denomination in these CSV exports is unconfirmed.
#
# Operation
# - This reads saved outputs; it does not repeat CSV ETL.
# - Two cells scan selected Parquet columns across the full history.
# - No quote cleaning or interpolation rules change here.
# - Fill Cell 12 after reading the report.
# - Then rerun Cells 12–15 to record the review without rescanning.

In [ ]:
# Settings and input checks.

from pathlib import Path
from datetime import datetime, timezone
from html import escape
import hashlib
import importlib.util
import json

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import HTML, display

RV_REPO = Path.cwd().resolve()
if RV_REPO.name == "notebooks":
    RV_REPO = RV_REPO.parent

assert (RV_REPO / "data").is_dir()
assert (RV_REPO / "notebooks").is_dir()

RV_RUN_ID = "etl_01_v3"
RV_TZ = "Europe/Athens"
RV_DETAIL_LIMIT = 10_000

RV_RUN = RV_REPO / "data" / "processed" / RV_RUN_ID
RV_REPORT = RV_REPO / "reports" / "tables" / "etl" / RV_RUN_ID

RV_CONFIG = json.loads(
    (RV_RUN / "configuration.json").read_text(encoding="utf-8")
)
RV_STATE = json.loads(
    (RV_RUN / "run_state.json").read_text(encoding="utf-8")
)
RV_SUMMARY = json.loads(
    (RV_REPORT / "execution_summary.json").read_text(encoding="utf-8")
)

assert RV_STATE["status"] == "complete"
assert RV_STATE["fingerprint"] == RV_SUMMARY["fingerprint"]
assert hashlib.sha256(
    json.dumps(RV_CONFIG, sort_keys=True).encode()
).hexdigest() == RV_SUMMARY["fingerprint"]

assert RV_SUMMARY["full_history_machine_audit_passed"]
assert all(value == 0 for value in RV_SUMMARY["final_checks"].values())

RV_AUDIT = (
    RV_REPO / "reports" / "tables" / "data_audit"
    / RV_CONFIG["audit_run"]
)

RV_OBSERVED_FILES = sorted((RV_RUN / "observed").glob("*.parquet"))
RV_ANALYSIS_FILES = sorted((RV_RUN / "analysis").glob("*.parquet"))

assert RV_OBSERVED_FILES
assert {p.stem for p in RV_OBSERVED_FILES} == {
    p.stem for p in RV_ANALYSIS_FILES
}
assert len(RV_OBSERVED_FILES) == RV_SUMMARY["months_processed"]

RV_LIMIT_NS = int(RV_CONFIG["gap_limit_seconds"]) * 1_000_000_000
RV_STEP_NS = int(RV_CONFIG["step_seconds"]) * 1_000_000_000

RV_CHECKS = {
    "observed_row_total_difference": abs(
        sum(pq.ParquetFile(p).metadata.num_rows for p in RV_OBSERVED_FILES)
        - RV_SUMMARY["observed_rows"]
    ),
    "analysis_row_total_difference": abs(
        sum(pq.ParquetFile(p).metadata.num_rows for p in RV_ANALYSIS_FILES)
        - RV_SUMMARY["analysis_rows"]
    )
}

RV_SECTIONS = {}

print("Reviewing:", RV_RUN_ID)
print("Observed months:", len(RV_OBSERVED_FILES))
print("DuckDB:", duckdb.__version__)

In [ ]:
# Save small report helpers in the functions folder.

RV_HELPER = RV_REPO / "functions" / "etl" / "output_validation.py"
RV_HELPER.parent.mkdir(parents=True, exist_ok=True)

RV_HELPER_SOURCE = r'''
import json
from html import escape
from pathlib import Path

import pandas as pd

STYLE = """
<style>
.rv-section {font-family:Arial,sans-serif;color:#17212b;background:white;
             padding:14px;margin:12px 0;border:1px solid #d6dee6}
.rv-section h2 {margin-top:0}
.rv-section p {line-height:1.45}
.rv-scroll {overflow:auto;max-height:520px;margin:10px 0}
.rv-section table {border-collapse:collapse;font-size:13px;width:100%}
.rv-section th,.rv-section td {border:1px solid #d6dee6;padding:6px;
                              text-align:left;vertical-align:top}
.rv-section th {background:#edf2f7;position:sticky;top:0}
.rv-section td {white-space:nowrap}
.rv-section details {margin:12px 0}
.rv-section summary {cursor:pointer;font-weight:bold}
</style>
"""

def times(frame, columns):
    result = frame.copy()
    for column in columns:
        result[column] = pd.to_datetime(
            result[column], unit="ns", utc=True
        ).dt.tz_convert("Europe/Athens").map(
            lambda value: "" if pd.isna(value) else value.isoformat()
        )
        result = result.rename(columns={
            column: column.removesuffix("_ns") + "_athens"
        })
    return result

def section(title, status, notes, tables):
    parts = [
        '<section class="rv-section">',
        f"<h2>{escape(title)}</h2>",
        f"<p><strong>Status:</strong> {escape(status)}</p>"
    ]
    parts.extend(f"<p>{escape(note)}</p>" for note in notes)

    for label, frame in tables:
        parts.append(
            f"<details {'open' if len(frame) <= 25 else ''}>"
            f"<summary>{escape(label)} — {len(frame):,} rows</summary>"
        )
        if frame.empty:
            parts.append("<p>No records.</p>")
        else:
            parts.append('<div class="rv-scroll">')
            parts.append(frame.to_html(
                index=False,
                escape=True,
                render_links=True,
                border=0,
                float_format=lambda value: f"{value:.9g}"
            ))
            parts.append("</div>")
        parts.append("</details>")

    parts.append("</section>")
    return "".join(parts)

def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False),
        encoding="utf-8"
    )
    temporary.replace(path)

def upsert_markdown(path, key, body):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = path.read_text(encoding="utf-8") if path.exists() else ""
    start = f"<!-- {key}:start -->"
    end = f"<!-- {key}:end -->"
    block = f"{start}\n{body.strip()}\n{end}"

    if start in text:
        left = text.index(start)
        right = text.index(end, left) + len(end)
        text = text[:left] + block + text[right:]
    else:
        text = text.rstrip() + "\n\n" + block + "\n"

    path.write_text(text, encoding="utf-8")
'''

RV_HELPER.write_text(
    RV_HELPER_SOURCE,
    encoding="utf-8",
    newline="\n"
)

rv_spec = importlib.util.spec_from_file_location(
    "fx_output_validation", RV_HELPER
)
rv = importlib.util.module_from_spec(rv_spec)
rv_spec.loader.exec_module(rv)

print("Report helpers loaded.")

In [ ]:
# EXPENSIVE: inspect quote-quality flags across the saved observed history.
# Include one-sided zero sizes, not only observations with both sizes zero.
# Only matching observations are materialized.

rv_db = duckdb.connect(config={
    "memory_limit": "2GB",
    "threads": "2"
})

rv_db.read_parquet(
    [str(p) for p in RV_OBSERVED_FILES],
    hive_partitioning=False
).create_view("rv_observed", replace=True)

rv_db.read_parquet(
    [str(p) for p in RV_ANALYSIS_FILES],
    hive_partitioning=False
).create_view("rv_analysis", replace=True)

rv_windows = json.loads(
    (RV_RUN / "merged_exclusion_windows.json").read_text(encoding="utf-8")
)

rv_db.register("rv_windows", pa.table({
    "start_ns": pa.array([row[0] for row in rv_windows], type=pa.int64()),
    "end_ns": pa.array([row[1] for row in rv_windows], type=pa.int64())
}))

# Local-day lookup avoids changing the original timestamp precision.
rv_first = pd.Timestamp(
    RV_SUMMARY["first_timestamp_athens"]
).tz_convert(RV_TZ)
rv_last = pd.Timestamp(
    RV_SUMMARY["last_timestamp_athens"]
).tz_convert(RV_TZ)

rv_days = pd.date_range(
    rv_first.normalize(),
    rv_last.normalize(),
    freq="D"
)

rv_db.register("rv_days", pa.table({
    "day_start_ns": pa.array(
        rv_days.as_unit("ns").asi8, type=pa.int64()
    ),
    "date_athens": pa.array(rv_days.strftime("%Y-%m-%d").tolist())
}))

rv_db.execute("""
    CREATE TEMP TABLE rv_flags AS
    SELECT
        q.*,
        e.end_ns IS NULL OR q.ts_ns >= e.end_ns AS in_analysis_scope,
        d.date_athens
    FROM (
        SELECT
            ts_ns, bid, ask, bid_size, ask_size,
            representative_source_id,
            representative_source_record,
            source_copies,
            ask = bid AS locked,
            bid_size = 0 AS bid_zero,
            ask_size = 0 AS ask_zero
        FROM rv_observed
        WHERE ask = bid OR bid_size = 0 OR ask_size = 0
    ) q
    ASOF LEFT JOIN rv_windows e ON q.ts_ns >= e.start_ns
    ASOF LEFT JOIN rv_days d ON q.ts_ns >= d.day_start_ns
""").complete()

rv_result = rv_db.sql("""
    SELECT
        count(*) AS flagged_observations,
        count(*) FILTER (WHERE locked) AS locked,
        count(*) FILTER (WHERE bid_zero AND ask_zero) AS both_zero,
        count(*) FILTER (WHERE bid_zero AND NOT ask_zero) AS bid_zero_only,
        count(*) FILTER (WHERE ask_zero AND NOT bid_zero) AS ask_zero_only,
        count(*) FILTER (
            WHERE (bid_zero OR ask_zero) AND in_analysis_scope
        ) AS zero_size_observations_in_analysis_scope
    FROM rv_flags
""")
RV_FLAG_METRICS = dict(zip(rv_result.columns, rv_result.fetchall()[0]))

RV_CHECKS["locked_count_difference"] = abs(
    RV_FLAG_METRICS["locked"] - RV_SUMMARY["locked_observed_rows"]
)
RV_CHECKS["both_zero_count_difference"] = abs(
    RV_FLAG_METRICS["both_zero"]
    - RV_SUMMARY["both_sizes_zero_observed_rows"]
)

print("Observed quote-quality scan complete.")
display(pd.Series(RV_FLAG_METRICS, name="observations").to_frame())

In [ ]:
# EXPENSIVE: read saved synthetic timestamps across the analysis history.
# Keep only the columns needed for boundary and availability review.
# Large query results remain inside DuckDB.

rv_db.execute("""
    CREATE TEMP TABLE rv_synthetic AS
    SELECT
        q.*,
        coalesce(q.left_anchor_ns < a.end_ns, false) AS left_excluded,
        coalesce(q.right_anchor_ns < b.end_ns, false) AS right_excluded
    FROM (
        SELECT
            ts_ns, available_ns, left_anchor_ns, right_anchor_ns,
            data_segment_id, scope_id
        FROM rv_analysis
        WHERE is_interpolated
    ) q
    ASOF LEFT JOIN rv_windows a
        ON q.left_anchor_ns >= a.start_ns
    ASOF LEFT JOIN rv_windows b
        ON q.right_anchor_ns >= b.start_ns
""").complete()

rv_result = rv_db.sql(f"""
    SELECT
        count(*) AS synthetic_rows,
        count(*) FILTER (
            WHERE available_ns IS NULL
               OR ts_ns IS NULL
               OR left_anchor_ns IS NULL
               OR right_anchor_ns IS NULL
        ) AS missing_timing_fields,
        count(*) FILTER (
            WHERE available_ns <> right_anchor_ns
        ) AS availability_not_equal_to_B,
        count(*) FILTER (
            WHERE available_ns < ts_ns
        ) AS availability_before_point,
        count(*) FILTER (
            WHERE NOT (
                left_anchor_ns < ts_ns
                AND ts_ns < right_anchor_ns
            )
        ) AS invalid_anchor_order,
        count(*) FILTER (
            WHERE right_anchor_ns - left_anchor_ns > {RV_LIMIT_NS}
        ) AS parent_gap_above_limit,
        count(*) FILTER (
            WHERE (ts_ns - left_anchor_ns) % {RV_STEP_NS} <> 0
        ) AS incorrect_ten_second_offset,
        min((available_ns - ts_ns) / 1e9) AS minimum_delay_seconds,
        avg((available_ns - ts_ns) / 1e9) AS mean_delay_seconds,
        max((available_ns - ts_ns) / 1e9) AS maximum_delay_seconds
    FROM rv_synthetic
""")

RV_SYNTHETIC_METRICS = dict(
    zip(rv_result.columns, rv_result.fetchall()[0])
)

for key in [
    "missing_timing_fields",
    "availability_not_equal_to_B",
    "availability_before_point",
    "invalid_anchor_order",
    "parent_gap_above_limit",
    "incorrect_ten_second_offset"
]:
    RV_CHECKS[key] = int(RV_SYNTHETIC_METRICS[key])

RV_CHECKS["synthetic_row_total_difference"] = abs(
    RV_SYNTHETIC_METRICS["synthetic_rows"]
    - RV_SUMMARY["synthetic_rows"]
)

print("Synthetic timestamp scan complete.")

In [ ]:
# Review 1: remaining dead zones.
# Raymond has accepted these as valid exclusions.
# Show every allowed-time piece and its original real-quote gap.
#
# A short allowed piece can belong to a raw gap above five minutes.
# Exclusions never make that original long gap eligible for interpolation.

rv_db.read_parquet(
    str(RV_REPORT / "dead_zone_pieces.parquet"),
    hive_partitioning=False
).create_view("rv_dead", replace=True)

rv_dead = rv_db.sql("""
    SELECT * FROM rv_dead ORDER BY piece_start_ns
""").df()

rv_dead.insert(
    0, "review_id",
    [f"D{index:03d}" for index in range(1, len(rv_dead) + 1)]
)
rv_dead["raw_gap_minutes"] = rv_dead["raw_gap_ns"] / 6e10
rv_dead["allowed_piece_minutes"] = rv_dead["piece_duration_ns"] / 6e10

RV_CHECKS["dead_piece_count_difference"] = abs(
    len(rv_dead) - RV_SUMMARY["dead_zone_pieces"]
)
RV_CHECKS["dead_parents_at_or_below_limit"] = int(
    (rv_dead["raw_gap_ns"] <= RV_LIMIT_NS).sum()
)

rv_dead_display = rv.times(rv_dead[[
    "review_id", "year_month",
    "a_ns", "b_ns",
    "piece_start_ns", "piece_end_ns",
    "raw_gap_minutes", "allowed_piece_minutes"
]], ["a_ns", "b_ns", "piece_start_ns", "piece_end_ns"])

rv_dead_display.to_csv(
    RV_REPORT / "01a_dead_zone_details.csv", index=False
)

rv_dead_monthly = pd.read_csv(RV_REPORT / "dead_zone_monthly.csv")

RV_SECTIONS["dead_zones"] = rv.section(
    "1. Remaining dead zones",
    "Accepted by Raymond",
    [
        "Raymond reviewed their dates and accepted the remaining periods "
        "as valid exclusions from the experiment.",
        "Their association with holidays, weekends and pre-holiday "
        "conditions is a domain interpretation, not proof of feed failure.",
        "No further answer is required for the existing exclusions."
    ],
    [
        ("Monthly summary", rv_dead_monthly),
        ("All dead-zone pieces", rv_dead_display)
    ]
)

display(HTML(rv.STYLE + RV_SECTIONS["dead_zones"]))

In [ ]:
# Review 2: locked quotes and zero displayed sizes.
#
# Locked means bid equals ask.
# Categories can overlap; do not add their counts together.
# Daily first/last times describe a span, not necessarily continuous flags.
#
# Raymond's zero-size decision:
# - Usable for price-based regime, trend, volatility and entropy analysis.
# - Excluded from P&L modeling inputs.
# - Cash and inventory must still carry through these periods.
# - Execution and valuation handling will be specified during accounting.

rv_flag_daily = rv_db.sql("""
    SELECT
        date_athens,
        count(*) AS flagged_observations,
        count(*) FILTER (WHERE locked) AS locked,
        count(*) FILTER (WHERE bid_zero AND ask_zero) AS both_zero,
        count(*) FILTER (
            WHERE bid_zero AND NOT ask_zero
        ) AS bid_zero_only,
        count(*) FILTER (
            WHERE ask_zero AND NOT bid_zero
        ) AS ask_zero_only,
        count(*) FILTER (
            WHERE in_analysis_scope
        ) AS flagged_inside_analysis_scope,
        min(ts_ns) AS first_ns,
        max(ts_ns) AS last_ns
    FROM rv_flags
    GROUP BY date_athens ORDER BY date_athens
""").df()

rv_flag_daily_display = rv.times(
    rv_flag_daily, ["first_ns", "last_ns"]
)

# The current completed run has a small locked-quote result.
assert RV_FLAG_METRICS["locked"] <= 100_000, (
    "Use a bounded export for a larger locked-quote result."
)

rv_locked = rv_db.sql("""
    SELECT
        ts_ns, bid, ask, bid_size, ask_size,
        bid_zero, ask_zero, in_analysis_scope,
        representative_source_id,
        representative_source_record,
        source_copies
    FROM rv_flags
    WHERE locked
    ORDER BY ts_ns
""").df()

rv_source_names = pd.DataFrame(RV_CONFIG["sources"])[[
    "source_id", "path"
]].rename(columns={"source_id": "representative_source_id"})

rv_locked = rv_locked.merge(
    rv_source_names,
    on="representative_source_id",
    how="left",
    validate="many_to_one"
)
rv_locked_display = rv.times(rv_locked, ["ts_ns"])

assert len(rv_locked_display) == RV_FLAG_METRICS["locked"]

rv_locked_display.to_csv(
    RV_REPORT / "01a_locked_quotes.csv", index=False
)
rv_flag_daily_display.to_csv(
    RV_REPORT / "01a_quote_quality_by_day.csv", index=False
)

rv_zero_examples = rv_db.sql(f"""
    SELECT
        ts_ns, bid, ask, bid_size, ask_size,
        locked, in_analysis_scope
    FROM rv_flags
    WHERE bid_zero OR ask_zero
    ORDER BY ts_ns
    LIMIT {RV_DETAIL_LIMIT}
""").df()
rv_zero_examples = rv.times(rv_zero_examples, ["ts_ns"])

RV_SECTIONS["quote_quality"] = rv.section(
    "2. Locked quotes and zero displayed sizes",
    "Zero-size use agreed; locked-quote review open",
    [
        "The detailed locked-quote CSV contains every locked quote "
        "in this completed run, including price, sizes and source provenance.",
        "Inspect the dates, prices and sizes. Choose either retain_flagged "
        "or investigate in Cell 12.",
        "Zero-size observations remain available for the agreed market "
        "analysis and are excluded from future P&L modeling inputs.",
        f"Zero-size detail shows up to {RV_DETAIL_LIMIT:,} observations; "
        "counts and daily summaries cover the full observed history."
    ],
    [
        (
            "Full-history counts",
            pd.Series(
                RV_FLAG_METRICS, name="observations"
            ).rename_axis("measure").reset_index()
        ),
        ("Dates and first/last flagged times", rv_flag_daily_display),
        ("All locked quotes", rv_locked_display),
        ("Zero-size observations", rv_zero_examples)
    ]
)

display(HTML(rv.STYLE + RV_SECTIONS["quote_quality"]))
print("Locked-quote report:", RV_REPORT / "01a_locked_quotes.csv")

In [ ]:
# Review 3: absolute displayed-size denomination.
#
# Official ITick documentation identifies size at the best bid/ask.
# Official BI5 documentation specifies millions of base-currency units.
# That BI5 specification does not prove whether these CSVs were rescaled.
#
# Keep the CSV scale unconfirmed until export-setting evidence is available.
# No conversion is applied here.

rv_size_evidence = pd.DataFrame([
    {
        "item": "Meaning of bid/ask size",
        "finding": "Available displayed size at the best bid/ask",
        "status": "Provider definition documented",
        "source":
            "https://www.dukascopy.com/client/javadoc3/com/dukascopy/api/ITick.html"
    },
    {
        "item": "Underlying BI5 scale",
        "finding": "Millions of base-currency units",
        "status": "Provider BI5 specification documented",
        "source":
            "https://www.dukascopy.com/wiki/en/development/data-export/"
    },
    {
        "item": "Scale in these CSV exports",
        "finding": "No export-scale evidence has been recorded",
        "status": "Unconfirmed",
        "source": "Original export settings or a matching reference export needed"
    }
])

RV_SECTIONS["size_units"] = rv.section(
    "3. Displayed-size denomination",
    "CSV denomination remains unconfirmed",
    [
        "For EURUSD, an unchanged million-base-unit scale would mean "
        "millions of EUR. That interpretation is conditional.",
        "Headers and plausible numeric values do not establish export scale.",
        "This limitation can remain open for price-based analysis. "
        "Resolve it before calculations requiring absolute displayed size.",
        "These fields are not executed trade volumes or guaranteed fills."
    ],
    [("Evidence and remaining limitation", rv_size_evidence)]
)

display(HTML(rv.STYLE + RV_SECTIONS["size_units"]))

In [ ]:
# Review 4: anchors inside excluded hours.
#
# Current behavior:
# - Synthetic timestamps inside exclusions are removed.
# - A or B may still be a real quote inside an exclusion.
# - Every retained synthetic point remains available only at B.
#
# This is the remaining interpolation policy decision.

rv_db.execute("""
    CREATE TEMP TABLE rv_boundary_parents AS
    SELECT
        left_anchor_ns, right_anchor_ns,
        left_excluded, right_excluded,
        count(*) AS retained_synthetic_points,
        min(ts_ns) AS first_point_ns,
        max(ts_ns) AS last_point_ns
    FROM rv_synthetic
    WHERE left_excluded OR right_excluded
    GROUP BY
        left_anchor_ns, right_anchor_ns,
        left_excluded, right_excluded
""").complete()

rv_boundary_counts = rv_db.sql("""
    SELECT
        CASE
            WHEN left_excluded AND right_excluded THEN 'Both anchors excluded'
            WHEN left_excluded THEN 'A excluded; B allowed'
            ELSE 'A allowed; B excluded'
        END AS anchor_case,
        count(*) AS parent_gaps,
        sum(retained_synthetic_points)::BIGINT AS synthetic_points
    FROM rv_boundary_parents
    GROUP BY anchor_case ORDER BY anchor_case
""").df()

rv_boundary_parent_count = rv_db.sql("""
    SELECT count(*) FROM rv_boundary_parents
""").fetchall()[0][0]

rv_boundary_details = rv_db.sql(f"""
    SELECT *,
        (right_anchor_ns - left_anchor_ns) / 1e9 AS raw_gap_seconds
    FROM rv_boundary_parents
    ORDER BY right_anchor_ns
    LIMIT {RV_DETAIL_LIMIT}
""").df()

rv_boundary_display = rv.times(rv_boundary_details, [
    "left_anchor_ns", "right_anchor_ns",
    "first_point_ns", "last_point_ns"
])

rv_boundary_display.to_csv(
    RV_REPORT / "01a_boundary_anchor_cases.csv", index=False
)

RV_SECTIONS["boundary_anchors"] = rv.section(
    "4. Anchors inside excluded hours",
    "Policy decision open",
    [
        f"Matching parent gaps: {rv_boundary_parent_count:,}. "
        f"The detail table shows up to {RV_DETAIL_LIMIT:,}.",
        "allow_current keeps the present treatment: retained points may "
        "use real anchors inside excluded hours.",
        "require_allowed_anchors would require both real anchors to be "
        "outside exclusions. That choice requires an analysis-output revision.",
        "When B is excluded, its retained synthetic points still become "
        "available at B, never earlier.",
        "Enter your choice in Cell 12. This notebook does not apply it."
    ],
    [
        ("Affected cases", rv_boundary_counts),
        ("Actual boundary-anchor dates and times", rv_boundary_display)
    ]
)

display(HTML(rv.STYLE + RV_SECTIONS["boundary_anchors"]))

In [ ]:
# Review 5: historical holiday quote closures.
#
# Raymond accepts the current holiday-related exclusion treatment.
# Preserve the distinction between:
# - Calendar holidays.
# - Verified quote-feed closure intervals.
#
# Exact historical feed-closure completeness remains a documented limitation.

rv_verified_path = RV_AUDIT / "verified_quote_closures_used.csv"

assert hashlib.sha256(
    rv_verified_path.read_bytes()
).hexdigest() == RV_CONFIG["verified_closure_sha256"]

rv_verified = pd.read_csv(rv_verified_path)
rv_calendar = pd.read_csv(RV_AUDIT / "calendar_events.csv")

rv_calendar_summary = (
    rv_calendar.loc[
        rv_calendar["nominal_year"].between(rv_first.year, rv_last.year)
    ]
    .groupby(["nominal_year", "currency", "kind"])
    .size()
    .reset_index(name="listed_calendar_dates")
)

rv_holiday_status = pd.DataFrame([
    {
        "item": "Current exclusion treatment",
        "status": "Accepted by Raymond after reviewing dates"
    },
    {
        "item": "Verified quote-feed closure intervals supplied",
        "status": str(len(rv_verified))
    },
    {
        "item": "Exact historical closure completeness",
        "status": "Unverified; accepted as a documented limitation"
    },
    {
        "item": "Use of calendar dates",
        "status": "Context; not automatic proof that EURUSD quotes were closed"
    }
])

RV_SECTIONS["holiday_closures"] = rv.section(
    "5. Holiday quote-closure coverage",
    "Treatment accepted; evidence limitation retained",
    [
        "Raymond reviewed the remaining patterns and considers them "
        "consistent with holidays and thin surrounding sessions.",
        "This records acceptance of the research treatment. It does not "
        "claim that exact Dukascopy closure/reopening times were verified.",
        "No further holiday investigation is required now."
    ],
    [
        ("Review status", rv_holiday_status),
        ("Supplied verified closure intervals", rv_verified),
        ("Supplied calendar context", rv_calendar_summary)
    ]
)

display(HTML(rv.STYLE + RV_SECTIONS["holiday_closures"]))

In [ ]:
# Review 6: right-anchor availability.
#
# These are separate checks:
# - A-to-B duration determines interpolation eligibility.
# - available_at = B determines information availability.
#
# A positive delay between the synthetic timestamp and availability is expected.
# A future feature cannot use the synthetic value before B arrives.

rv_availability_table = pd.DataFrame([
    {"check": key, "error_count": value}
    for key, value in RV_CHECKS.items()
])

rv_availability_table["result"] = np.where(
    rv_availability_table["error_count"] == 0,
    "PASS",
    "FAIL"
)

rv_delay_table = pd.DataFrame([
    {
        "measure": key,
        "value": RV_SYNTHETIC_METRICS[key]
    }
    for key in [
        "synthetic_rows",
        "minimum_delay_seconds",
        "mean_delay_seconds",
        "maximum_delay_seconds"
    ]
])

rv_example_file = RV_REPORT / "example_interpolated_points.csv"
if rv_example_file.exists():
    rv_timing_example = pd.read_csv(
        rv_example_file,
        dtype={"ts_ns": "int64", "available_ns": "int64"}
    )
    rv_timing_example["availability_delay_seconds"] = (
        rv_timing_example["available_ns"] - rv_timing_example["ts_ns"]
    ) / 1e9

    rv_timing_example = rv.times(
        rv_timing_example.drop(columns=[
            name for name in ["timestamp_athens", "available_at_athens"]
            if name in rv_timing_example.columns
        ]),
        ["ts_ns", "available_ns"]
    )
else:
    rv_timing_example = pd.DataFrame()

RV_MACHINE_PASSED = all(value == 0 for value in RV_CHECKS.values())

RV_SECTIONS["availability"] = rv.section(
    "6. Right-anchor availability",
    "PASS" if RV_MACHINE_PASSED else "FAIL — resolve before proceeding",
    [
        "Raymond reviewed examples showing right anchor B.",
        "Notebook 01 reported passing interpolation checks. "
        "This notebook checks saved synthetic timing again.",
        "The required rule remains available_at = B. "
        "Human approval cannot waive that timing requirement.",
        "Future feature calculations must respect availability, "
        "even when the synthetic point has an earlier timestamp."
    ],
    [
        ("Saved-output validation checks", rv_availability_table),
        ("Availability delays", rv_delay_table),
        ("Saved interpolation example", rv_timing_example)
    ]
)

display(HTML(rv.STYLE + RV_SECTIONS["availability"]))

rv_db.close()
print("Evidence collection complete. The database connection is closed.")

In [ ]:
# Reconcile the recorded answers with Raymond's latest locked-quote decision.
# Preserve the existing size-unit and boundary-anchor answers.
# No Parquet scan or quote-data change is required.

rv_saved_answers = json.loads(
    (RV_REPORT / "01a_review_answers.json").read_text(encoding="utf-8")
)
assert rv_saved_answers["fingerprint"] == RV_SUMMARY["fingerprint"]

RV_ANSWERS = rv_saved_answers["answers"]

LOCKED_QUOTE_DECISION = "retain_flagged"
LOCKED_QUOTE_NOTES = (
    "Retain locked quotes for regime, trend, entropy and volatility "
    "analysis. Exclude them from P&L simulation inputs. "
    "The recorded flag means bid equals ask; it does not independently "
    "establish that prices were stuck."
)

rv_quality_answer = RV_ANSWERS["locked_quotes_and_zero_sizes"]
rv_quality_answer.setdefault(
    "original_locked_quote_notes",
    rv_quality_answer.get("locked_quote_notes", "")
)
rv_quality_answer.update({
    "locked_quote_decision": LOCKED_QUOTE_DECISION,
    "locked_quote_notes": LOCKED_QUOTE_NOTES,
    "agreed_locked_quote_analysis_use": "retain_with_flags",
    "agreed_locked_quote_pnl_input_use": "exclude",
    "agreed_zero_size_analysis_use": "retain_with_flags",
    "agreed_zero_size_pnl_input_use": "exclude"
})

BOUNDARY_ANCHOR_DECISION = (
    RV_ANSWERS["excluded_hour_anchors"]["decision"]
)
BOUNDARY_ANCHOR_NOTES = (
    RV_ANSWERS["excluded_hour_anchors"].get("notes", "")
)

CSV_SIZE_DENOMINATION = (
    RV_ANSWERS["size_denomination"]["denomination"]
)
CSV_SIZE_EVIDENCE = (
    RV_ANSWERS["size_denomination"].get("evidence", "")
)

RV_OPEN_DECISIONS = [
    name for name, value in {
        "locked_quote_decision": LOCKED_QUOTE_DECISION,
        "boundary_anchor_decision": BOUNDARY_ANCHOR_DECISION
    }.items() if not value
]

RV_REVIEW_READY = not RV_OPEN_DECISIONS and RV_MACHINE_PASSED
RV_CHANGES_REQUESTED = (
    LOCKED_QUOTE_DECISION == "investigate"
    or BOUNDARY_ANCHOR_DECISION == "require_allowed_anchors"
)

# Refresh report sections that previously displayed pending decisions.

RV_SECTIONS["quote_quality"] = rv.section(
    "2. Locked quotes and zero displayed sizes",
    "Treatment accepted by Raymond",
    [
        "Retain both groups with flags for the agreed market analysis.",
        "Exclude locked quotes and zero-size observations from "
        "P&L simulation inputs.",
        "Cash and inventory must still carry through those periods. "
        "Execution and valuation handling will be designed during accounting."
    ],
    [
        (
            "Full-history counts",
            pd.Series(
                RV_FLAG_METRICS, name="observations"
            ).rename_axis("measure").reset_index()
        ),
        ("Dates and first/last flagged times", rv_flag_daily_display),
        ("All locked quotes", rv_locked_display),
        ("Zero-size observations", rv_zero_examples)
    ]
)

rv_unit_review = rv_size_evidence.loc[
    rv_size_evidence["item"] != "Scale in these CSV exports"
].copy()

rv_unit_review = pd.concat([
    rv_unit_review,
    pd.DataFrame([{
        "item": "Scale in these CSV exports",
        "finding": "Millions of EUR",
        "status": "Confirmed by Raymond Fendi",
        "source": CSV_SIZE_EVIDENCE
    }])
], ignore_index=True)

RV_SECTIONS["size_units"] = rv.section(
    "3. Displayed-size denomination",
    "Millions of EUR — user confirmed",
    [
        "Raymond confirms that the CSV displayed-size values are "
        "in millions of EUR.",
        "A stored size of 1.0 therefore represents EUR 1,000,000 "
        "of displayed size. Stored values have not been rescaled.",
        "This records Raymond's confirmation; it does not claim an "
        "independent verification of the original exporter settings.",
        "Displayed size is not executed trade volume or a guaranteed fill."
    ],
    [("Unit definition and evidence", rv_unit_review)]
)

RV_SECTIONS["boundary_anchors"] = rv.section(
    "4. Anchors inside excluded hours",
    "Current treatment accepted by Raymond",
    [
        "Excluded-hour quotes may remain real interpolation anchors.",
        "Synthetic timestamps inside excluded hours remain excluded.",
        "Synthetic observations retain availability at right anchor B.",
        BOUNDARY_ANCHOR_NOTES,
        f"This run contains {rv_boundary_parent_count:,} affected parent gaps."
    ],
    [
        ("Affected cases", rv_boundary_counts),
        ("Boundary-anchor dates and times", rv_boundary_display)
    ]
)

print("Review responses complete:", RV_REVIEW_READY)
print("Changes requested:", RV_CHANGES_REQUESTED)
print("Next: run Cells 13, 14 and 15.")

In [ ]:
# Save one combined HTML report and the review answers.
# This also works before the two open decisions are answered.

rv_section_order = [
    "dead_zones",
    "quote_quality",
    "size_units",
    "boundary_anchors",
    "holiday_closures",
    "availability"
]
assert all(key in RV_SECTIONS for key in rv_section_order)

rv_answer_table = pd.DataFrame([
    {
        "review_item": key,
        "recorded_answer": json.dumps(value, ensure_ascii=False)
    }
    for key, value in RV_ANSWERS.items()
])

rv_answer_block = rv.section(
    "Recorded review answers",
    "Open decisions remain" if RV_OPEN_DECISIONS else "Responses recorded",
    [
        "Recorded answers do not automatically change the dataset.",
        "A requested change must be implemented and validated before "
        "affected outputs are reused."
    ],
    [("Answers", rv_answer_table)]
)

RV_HTML_PATH = RV_REPORT / "01a_output_validation.html"
RV_ANSWER_PATH = RV_REPORT / "01a_review_answers.json"
RV_VALIDATION_PATH = RV_REPORT / "01a_validation_summary.json"

rv_generated = datetime.now(timezone.utc).isoformat()

rv_document = (
    "<!doctype html><html><head><meta charset='utf-8'>"
    "<title>Notebook 01a output validation</title>"
    + rv.STYLE
    + "</head><body>"
    + "<section class='rv-section'>"
    + "<h1>Notebook 01a — output validation</h1>"
    + f"<p>ETL run: {escape(RV_RUN_ID)}</p>"
    + f"<p>Generated: {escape(rv_generated)}</p>"
    + "<p>Displayed timestamps use Europe/Athens and include UTC offsets.</p>"
    + "</section>"
    + "".join(RV_SECTIONS[key] for key in rv_section_order)
    + rv_answer_block
    + "</body></html>"
)

RV_HTML_PATH.write_text(rv_document, encoding="utf-8")

rv.save_json(RV_ANSWER_PATH, {
    "etl_run_id": RV_RUN_ID,
    "fingerprint": RV_SUMMARY["fingerprint"],
    "recorded_at_utc": rv_generated,
    "answers": RV_ANSWERS
})

RV_VALIDATION_SUMMARY = {
    "notebook": "01a_output_validation.ipynb",
    "etl_run_id": RV_RUN_ID,
    "fingerprint": RV_SUMMARY["fingerprint"],
    "machine_checks_passed": RV_MACHINE_PASSED,
    "checks": RV_CHECKS,
    "locked_quotes_reported": len(rv_locked_display),
    "boundary_parent_gaps": rv_boundary_parent_count,
    "open_decisions": RV_OPEN_DECISIONS,
    "csv_size_denomination": CSV_SIZE_DENOMINATION,
    "changes_requested": RV_CHANGES_REQUESTED,
    "review_responses_complete": RV_REVIEW_READY,
    "data_changes_applied": False
}
rv.save_json(RV_VALIDATION_PATH, RV_VALIDATION_SUMMARY)

display(HTML(rv.STYLE + rv_answer_block))
print("Combined report:", RV_HTML_PATH)
print("Locked quotes:", RV_REPORT / "01a_locked_quotes.csv")
print("Answers:", RV_ANSWER_PATH)

In [ ]:
# Record contributions and project decisions once the open answers are supplied.
# If a change is requested, record it as pending implementation.
# Do not mark changed data as already corrected.

if not RV_REVIEW_READY:
    print("Contributions will be recorded after these decisions are supplied:")
    print(RV_OPEN_DECISIONS)
    if not RV_MACHINE_PASSED:
        print("Machine-check failures must also be resolved.")
else:
    rv_review_status = (
        "changes_requested" if RV_CHANGES_REQUESTED else "accepted"
    )

    rv_contribution_text = f"""
## Notebook 01 output review — {RV_RUN_ID}

Raymond Fendi:
- Reviewed the remaining {RV_SUMMARY['dead_parent_intervals']} dead-zone
  intervals and accepted their exclusion from the experiment.
- Interpreted the dates as consistent with holidays, weekends and thin
  pre-holiday conditions. This is domain interpretation rather than
  independently verified evidence of each gap's cause.
- Accepted the holiday-related research treatment while retaining the
  limitation around exact historical provider closure coverage.
- Reviewed interpolation examples showing right anchor B.
- Agreed that synthetic observations must retain availability at B.
- Approved zero-size observations for price-based regime, trend,
  volatility and entropy analysis, and excluded them from P&L modeling inputs.
- Supplied the locked-quote decision: `{LOCKED_QUOTE_DECISION}`.
- Supplied the excluded-hour anchor decision: `{BOUNDARY_ANCHOR_DECISION}`.

Locked-quote notes: {LOCKED_QUOTE_NOTES or 'None supplied.'}

Boundary-anchor notes: {BOUNDARY_ANCHOR_NOTES or 'None supplied.'}

CSV size denomination: `{CSV_SIZE_DENOMINATION}`.

Size-unit evidence: {CSV_SIZE_EVIDENCE or 'No additional evidence supplied.'}

OpenAI Codex:
- Supplied the ETL implementation, compatibility corrections and validation code.
- Supplied Notebook 01a and the detailed review reports.
- Distinguished measured checks, user interpretation and unresolved evidence.

Machine validation passed: `{RV_MACHINE_PASSED}`.
Human review status: `{rv_review_status}`.
Requested data changes are not implemented by Notebook 01a.
"""

    rv.upsert_markdown(
        RV_REPO / "CONTRIBUTIONS.md",
        f"notebook-01-review-{RV_RUN_ID}",
        rv_contribution_text
    )

    rv_decision_text = f"""
## ETL output review — {RV_RUN_ID}

- Decision: retain the reviewed dead-zone exclusions.
  Rationale: Raymond's review of their dates and market context.
  Component: ETL analysis scope.
  Status: agreed.

- Decision: accept the current holiday treatment with exact historical
  feed-closure coverage retained as a limitation.
  Component: calendar interpretation.
  Status: agreed limitation.

- Decision: use zero-size observations for price-based market analysis;
  exclude them from P&L modeling inputs.
  Component: analysis and future accounting.
  Status: agreed; operational execution and valuation rules remain to be designed.

- Decision: synthetic availability must equal right anchor B.
  Component: feature timing and replay.
  Status: agreed and machine checked.

- Locked-quote review: `{LOCKED_QUOTE_DECISION}`.
  Notes: {LOCKED_QUOTE_NOTES or 'None supplied.'}
  Status: {'follow-up required' if LOCKED_QUOTE_DECISION == 'investigate' else 'retain with flags'}.

- Boundary-anchor decision: `{BOUNDARY_ANCHOR_DECISION}`.
  Notes: {BOUNDARY_ANCHOR_NOTES or 'None supplied.'}
  Status: {'analysis revision required' if BOUNDARY_ANCHOR_DECISION == 'require_allowed_anchors' else 'current treatment accepted'}.

- CSV size denomination: `{CSV_SIZE_DENOMINATION}`.
  Evidence: {CSV_SIZE_EVIDENCE or 'Unconfirmed; no additional evidence supplied.'}
  Component: size-dependent calculations.
  Status: {'unconfirmed limitation' if CSV_SIZE_DENOMINATION == 'unconfirmed' else 'user-supplied interpretation; supporting evidence recorded above'}.

- Dependency impact: if allowed anchors are required, the current
  analysis output must be revised before downstream reuse.
  The consolidated observed quotes do not require that interpolation revision.
"""

    rv.upsert_markdown(
        RV_REPO / "docs" / "design_decisions.md",
        f"notebook-01-review-{RV_RUN_ID}",
        rv_decision_text
    )

    if BOUNDARY_ANCHOR_DECISION == "require_allowed_anchors":
        rv_next_step = "Revise and validate the analysis interpolation output."
        rv_invalidated = (
            f"`data/processed/{RV_RUN_ID}/analysis/` is unsuitable for the "
            "newly requested anchor policy until revised. Regenerate any "
            "dependent outputs before reuse."
        )
    elif LOCKED_QUOTE_DECISION == "investigate":
        rv_next_step = "Investigate the selected locked-quote cases before advancing."
        rv_invalidated = "No additional outputs invalidated yet; investigation pending."
    else:
        rv_next_step = (
            "Discuss Notebook 02 sampling, candles and indicators before building."
        )
        rv_invalidated = "No additional outputs invalidated by this review."

    rv_project_state = f"""# Project state

- Current stage: Notebook 01a output review.
- ETL run: `{RV_RUN_ID}`.
- Notebook 00: full original-history audit executed.
- Notebook 01: full-history machine validation passed.
- Notebook 01a: review reports generated.
- Human ETL review: `{rv_review_status}`.
- Size denomination: `{CSV_SIZE_DENOMINATION}`.

## Locations

- Original CSVs: `data/`.
- Downloaded exports: `data/2017 2018 investigation/`.
- Observed quotes: `data/processed/{RV_RUN_ID}/observed/`.
- Analysis quotes: `data/processed/{RV_RUN_ID}/analysis/`.
- Review report: `reports/tables/etl/{RV_RUN_ID}/01a_output_validation.html`.
- Review answers: `reports/tables/etl/{RV_RUN_ID}/01a_review_answers.json`.

## Outstanding matters

- CSV size denomination requires evidence before absolute size-dependent use.
- Exact historical provider closure completeness remains a documented limitation.
- Zero-size observations are excluded from P&L modeling inputs.
- Execution and valuation handling will be designed during accounting.
- Locked-quote decision: `{LOCKED_QUOTE_DECISION}`.
- Boundary-anchor decision: `{BOUNDARY_ANCHOR_DECISION}`.

## Outputs affected

{rv_invalidated}

## Next step

{rv_next_step}
"""
    (RV_REPO / "docs" / "project_state.md").write_text(
        rv_project_state, encoding="utf-8"
    )

    rv_updated_state = json.loads(
        (RV_RUN / "run_state.json").read_text(encoding="utf-8")
    )
    assert rv_updated_state["fingerprint"] == RV_SUMMARY["fingerprint"]

    rv_updated_state["human_review"] = rv_review_status
    rv_updated_state["analysis_revision_required"] = (
        BOUNDARY_ANCHOR_DECISION == "require_allowed_anchors"
    )
    rv_updated_state["review_answers"] = (
        RV_ANSWER_PATH.relative_to(RV_REPO).as_posix()
    )
    rv.save_json(RV_RUN / "run_state.json", rv_updated_state)

    print("CONTRIBUTIONS.md and project records updated.")
    print("Human review status:", rv_review_status)
    print("Next step:", rv_next_step)

In [ ]:
# Return this summary and your two remaining decisions.
#
# To enter or revise answers:
# - Edit Cell 12.
# - Rerun Cells 12–15 only.
# - The saved Parquet scans do not need to be repeated.
#
# Stop here before building the next component.

print("NOTEBOOK_01A_EXECUTION_SUMMARY")
print(json.dumps(RV_VALIDATION_SUMMARY, indent=2))

print("\nRemaining decisions:")
print("Locked quotes:", LOCKED_QUOTE_DECISION or "Pending")
print("Excluded-hour anchors:", BOUNDARY_ANCHOR_DECISION or "Pending")

assert RV_MACHINE_PASSED, (
    "Review the reported validation failures before proceeding."
)

In [ ]:
# Reconcile documentation with the completed review.
# No data scan, quote change or ETL rerun.

assert RV_MACHINE_PASSED
assert RV_REVIEW_READY
assert not RV_CHANGES_REQUESTED

rv_final_state = json.loads(
    (RV_RUN / "run_state.json").read_text(encoding="utf-8")
)
assert rv_final_state["fingerprint"] == RV_SUMMARY["fingerprint"]
assert rv_final_state["human_review"] == "accepted"

rv_document_edits = {
    "CONTRIBUTIONS.md": [
        (
            "Execution and human review are pending.\n"
            "No research result or user validation is claimed.",
            "Notebook 01 and Notebook 01a machine checks passed. "
            "Raymond accepted the ETL outputs for the agreed market analysis. "
            "Trading results remain pending simulation and evaluation."
        ),
        (
            "- Raymond's review and acceptance of these outputs remain pending.",
            "- Raymond's subsequent acceptance is recorded in the "
            "Notebook 01 output review below."
        )
    ],
    "docs/project_state.md": [
        (
            "- Current stage: Notebook 01a output review.",
            "- Current stage: Notebook 01 and Notebook 01a review complete."
        ),
        (
            "- Size denomination: `millions`.",
            "- Size denomination: millions of EUR, confirmed by Raymond."
        ),
        (
            "- CSV size denomination requires evidence before absolute "
            "size-dependent use.",
            "- CSV size denomination is confirmed by Raymond as millions of EUR."
        ),
        (
            "- Zero-size observations are excluded from P&L modeling inputs.",
            "- Locked quotes and zero-size observations are excluded "
            "from P&L modeling inputs."
        )
    ],
    "README.md": [
        (
            "Human review of Notebook 01 remains pending.",
            "Raymond accepted Notebook 01 outputs for the agreed market "
            "analysis. Notebook 01a records the review and limitations."
        )
    ]
}

for relative_path, replacements in rv_document_edits.items():
    path = RV_REPO / relative_path
    text = path.read_text(encoding="utf-8")
    updated = text
    for old, new in replacements:
        updated = updated.replace(old, new)
    if updated != text:
        path.write_text(updated, encoding="utf-8")

rv.upsert_markdown(
    RV_REPO / "docs" / "design_decisions.md",
    f"notebook-01-review-{RV_RUN_ID}",
    f"""
## Accepted ETL output review — {RV_RUN_ID}

This review supersedes earlier pending or provisional review labels
for the same ETL run.

- Reviewed dead-zone exclusions: accepted by Raymond.
- Holiday treatment: accepted with exact historical provider closure
  completeness retained as a documented limitation.
- Locked quotes: retained with flags for regime, trend, entropy and
  volatility analysis; excluded from P&L simulation inputs.
- Zero-size observations: retained for agreed market analysis;
  excluded from P&L simulation inputs.
- Displayed-size denomination: millions of EUR, confirmed by Raymond.
- Real anchors inside excluded hours: permitted.
- Generated timestamps inside exclusions: remain excluded.
- Synthetic availability: must equal right anchor B.
- Machine validation: passed.
- Human review: accepted.
- Analysis regeneration: not required.

Locked means bid equals ask; that flag alone does not establish stuck prices.
P&L execution and valuation handling will be designed during accounting.
"""
)

rv_final_state.update({
    "user_confirmed_size_unit": "million_EUR",
    "agreed_pnl_input_exclusions": [
        "locked_quotes",
        "zero_displayed_sizes"
    ]
})
rv.save_json(RV_RUN / "run_state.json", rv_final_state)

print("Documentation reconciled.")
print("ETL review complete and accepted.")
print("Next: Notebook 02 design discussion.")